In [1]:
import sys
sys.path.append('..')

import numpy as np
from PIL import Image
from src.elements import *
from src.systems import *
from src.utilities import *

In [15]:
ThinL1 = ThinLens(100)
FreeS = FreeSpace(600)
ThinL2 = ThinLens(500)
System = OpticalSystem()
System.add_element(ThinL1)
System.add_element(FreeS)
System.add_element(ThinL2)
print("System matrix:", System.M)

System matrix: [[-5.e+00  6.e+02]
 [ 0.e+00 -2.e-01]]


In [2]:
ThickL1 = ThickLens(51.68, -51.68, 6, 1.5)
FreeS2 = FreeSpace(200)
ThickL2 = ThickLens(155.04, -155.04, 4, 1.5)
System2 = OpticalSystem(color=True)  # Create a new optical system that can handle color
System2.add_element(ThickL1)
System2.add_element(FreeS2)
System2.add_element(ThickL2)
print("System2 matrix:", System2.MR)

System2 matrix: [[-2.96752455e+00  1.96894991e+02]
 [ 9.15011212e-05 -3.43052297e-01]]


In [16]:
img_path = "../assets/PIA01464.jpg"  # Path to the input image
img = Image.open(img_path)
image_array = np.array(img)
object = Object(image_array, distance=200, height=20)  # Create an object with the image array, distance, and height
#A, B, C, D = System2.MG.flatten()  # Unpack the system matrix elements
A, B, C, D = System.M.flatten()  # Unpack the system matrix elements
d_img = -(A * object.distance + B) / D  # Calculate the image distance using the system matrix
print(f"Image distance for the system with thin lenses: {d_img:.2f} mm")
image_plane_propagation = FreeSpace(d_img)  # Create a free space propagation element with the calculated image distance
System.add_element(image_plane_propagation)  # Add a free space element to the system with the calculated image distance
print("Updated system matrix after adding free space propagation:", System.M)

Image distance for the system with thin lenses: -2000.00 mm
Updated system matrix after adding free space propagation: [[-5.e+00  1.e+03]
 [ 0.e+00 -2.e-01]]


In [14]:

print("Original image shape:", image_array.shape)
pixel_size = 0.01  # Example pixel size in mm
pupil_radius = 40  # Example pupil radius in mm
n_rays_per_pixel = 99  # Example number of rays per pixel
output_image_array = System.image_object(object, pupil_radius, pixel_size, n_rays_per_pixel, interpolation=True)  # Process the image through the optical system
output_image = Image.fromarray(output_image_array)
print("Output image shape:", output_image.size)
output_image.save("../results/saturn_output.jpg")

Original image shape: (256, 371, 3)
Output image shape: (157, 150)


In [6]:
# Interpolate the image through the system
interpolated_image_array = System.image_with_interpolation(image_array, pixel_size)
interpolated_image = Image.fromarray(interpolated_image_array)
print("Interpolated image shape:", interpolated_image.size)
interpolated_image.save("../results/saturn_interpolated_output2.jpg")

Interpolated image shape: (1114, 662)


In [6]:
Lens1 = ThickLens(150, -150, 8, 1.5)
Lens2 = ThickLens(-52, 52, 5, 1.5)
Gal_Thick = GalileanTelescope(lens1=Lens1, lens2=Lens2, color=True)  
Gal_Thin_image_array = Gal_Thick.image_infinity(image_array)
Gal_Thick_image_array = Gal_Thick.image_with_interpolation(image_array, infinity=True)
Gal_Thick_image = Image.fromarray(Gal_Thick_image_array)
print("Galilean telescope with thick lenses output image shape:", Gal_Thick_image.size)
Gal_Thick_image.save("../results/saturn_galilean_thick_output.jpg")
print("Galilean telescope with thick lenses system matrix M:")
print(Gal_Thick.MRGB)

Galilean telescope with thick lenses output image shape: (799, 642)
Galilean telescope with thick lenses system matrix M:
[array([[ 2.88609718e-01,  1.10260402e+02],
       [-9.61718619e-04,  3.09747199e+00]]), array([[ 2.85145968e-01,  1.10250805e+02],
       [-1.03352982e-03,  3.10736467e+00]]), array([[ 2.77341782e-01,  1.10229289e+02],
       [-1.19766935e-03,  3.12964658e+00]])]


In [ ]:
R1_o = 153.7
R2_o = -153.7
R3_o = -706.8
d1_o = 8.4
d2_o = 7
f_o = 300

R1_e = -42.0
R2_e = 39.5
R3_e = 207.2
d1_e = 2
d2_e = 4.5
f_e = -50

n_reference = refractive_index_NBK7(0.5876)
n_ref_NSF2 = refractive_index_NSF2(0.5876)

Doublet1 = Doublet(R1_o, R2_o, R3_o, d1_o, d2_o, n_reference, n_ref_NSF2)
Doublet2 = Doublet(R1_e, R2_e, R3_e, d1_e, d2_e, n_reference, n_ref_NSF2)

Doublet_telescope = GalileanTelescope(lens1=Doublet1, lens2=Doublet2, color=True, material1=['NBK7', 'NSF2'], material2=['NBAF10', 'NSF6HT'])
Doublet_image_array = Doublet_telescope.image_infinity(image_array)
Doublet_image = Image.fromarray(Doublet_image_array)
print("Galilean telescope with doublet lenses output image shape:", Doublet_image.size)
Doublet_image.save("../results/saturn_doublet_output.jpg")
print("Galilean telescope with doublet lenses system matrix M:")
print(Doublet_telescope.MRGB)   

Galilean telescope with doublet lenses output image shape: (1185, 952)
Galilean telescope with doublet lenses system matrix M:
[array([[2.51779005e-01, 2.34348945e+02],
       [7.12813121e-04, 4.63520382e+00]]), array([[2.50896322e-01, 2.34287677e+02],
       [7.00358058e-04, 4.63970636e+00]]), array([[2.50025574e-01, 2.34125668e+02],
       [6.88473424e-04, 4.64428211e+00]])]


In [ ]:
# My telescope

image_path = "../assets/jupiter-through-a-16-telescope-v0-8w9mx1hyqxwa1.webp"
image_array = np.array(Image.open(image_path))
obj_lens = ThickLens(206.72, -np.inf, 8.5, refractive_index(0.5876, 'NBK7'))
eye_lens = ThickLens(5.17, -np.inf, 3.7, refractive_index(0.5876, 'NBK7'))
My_telescope = GalileanTelescope(lens1=obj_lens, lens2=eye_lens, color=True)
My_image_array = My_telescope.image_with_interpolation(image_array, infinity=True)
My_image = Image.fromarray(My_image_array)
print("My Galilean telescope output image shape:", My_image.size)
My_image.save("../results/my_galilean_telescope_output.jpg")
print("My Galilean telescope system matrix M:")
print(My_telescope.MRGB)   

My Galilean telescope output image shape: (24472, 22053)
My Galilean telescope system matrix M:
[array([[-3.18619153e-02,  3.17038047e+02],
       [ 9.00574910e-04, -4.03464920e+01]]), array([[-3.56032755e-02,  3.16706368e+02],
       [ 1.40042675e-03, -4.05446985e+01]]), array([[-4.40038350e-02,  3.15963149e+02],
       [ 2.54387385e-03, -4.09912089e+01]])]
